# Bài 4 - Neural Network trên Iris bằng 10-fold Cross Validation

**Sinh viên:** Nguyễn Đức Vinh  
**MSSV:** 6451071087

Huấn luyện Neural Network 10 fold, lưu accuracy/precision/recall/F1-score của từng fold, tính trung bình và dự đoán 3 mẫu mới.

> Chạy lần lượt các cell từ trên xuống dưới trong Google Colab. Mỗi notebook là một bài độc lập.

## 1. Thư viện và thông tin sinh viên

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.datasets import load_iris
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print("Nguyễn Đức Vinh - MSSV: 6451071087")
tf.keras.utils.set_random_seed(42)

Nguyễn Đức Vinh - MSSV: 6451071087


## 2. Đọc và chuẩn hóa Iris (câu a, b)

In [2]:
print("Nguyễn Đức Vinh - MSSV: 6451071087")
iris = load_iris()
X_raw = iris.data.astype("float32")
y = iris.target
class_names = iris.target_names
feature_max = X_raw.max(axis=0)
X = X_raw / feature_max
print("X:", X.shape, "y:", y.shape)
print("Classes:", class_names)

Nguyễn Đức Vinh - MSSV: 6451071087
X: (150, 4) y: (150,)
Classes: ['setosa' 'versicolor' 'virginica']


## 3. Định nghĩa create_model (câu c)

In [3]:
print("Nguyễn Đức Vinh - MSSV: 6451071087")
def create_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(X.shape[1],)),
        tf.keras.layers.Dense(10, activation="relu"),
        tf.keras.layers.Dense(20, activation="relu"),
        tf.keras.layers.Dense(len(class_names), activation="softmax")
    ])
    model.compile(optimizer="adam",
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

Nguyễn Đức Vinh - MSSV: 6451071087


## 4. Huấn luyện 10-fold và lưu chỉ số từng fold (câu d)

In [4]:
print("Nguyễn Đức Vinh - MSSV: 6451071087")
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
models = []
fold_rows = []
per_class_rows = []

for fold, (train_index, val_index) in enumerate(kfold.split(X), start=1):
    tf.keras.backend.clear_session()
    model = create_model()
    model.fit(X[train_index], y[train_index], epochs=100,
              batch_size=1, verbose=0)
    probabilities = model.predict(X[val_index], verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    y_true = y[val_index]

    accuracy = accuracy_score(y_true, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, predictions, labels=np.arange(len(class_names)),
        average="macro", zero_division=0
    )
    class_p, class_r, class_f1, _ = precision_recall_fscore_support(
        y_true, predictions, labels=np.arange(len(class_names)),
        average=None, zero_division=0
    )
    fold_rows.append({"Fold": fold, "Accuracy": accuracy,
                      "Precision": precision, "Recall": recall, "F1-score": f1})
    for class_index, class_name in enumerate(class_names):
        per_class_rows.append({"Fold": fold, "Class": class_name,
                               "Precision": class_p[class_index],
                               "Recall": class_r[class_index],
                               "F1-score": class_f1[class_index]})
    models.append(model)
    print(f"Fold {fold:02d}: Accuracy={accuracy:.4f}, Precision={precision:.4f}, "
          f"Recall={recall:.4f}, F1={f1:.4f}")

Nguyễn Đức Vinh - MSSV: 6451071087
Fold 01: Accuracy=1.0000, Precision=1.0000, Recall=1.0000, F1=1.0000
Fold 02: Accuracy=1.0000, Precision=1.0000, Recall=1.0000, F1=1.0000
Fold 03: Accuracy=1.0000, Precision=1.0000, Recall=1.0000, F1=1.0000
Fold 04: Accuracy=0.9333, Precision=0.9524, Recall=0.9333, F1=0.9373


Fold 05: Accuracy=1.0000, Precision=1.0000, Recall=1.0000, F1=1.0000


Fold 06: Accuracy=0.9333, Precision=0.9167, Recall=0.9444, F1=0.9221
Fold 07: Accuracy=0.8667, Precision=0.9167, Recall=0.8333, F1=0.8413
Fold 08: Accuracy=1.0000, Precision=1.0000, Recall=1.0000, F1=1.0000
Fold 09: Accuracy=1.0000, Precision=1.0000, Recall=1.0000, F1=1.0000
Fold 10: Accuracy=1.0000, Precision=1.0000, Recall=1.0000, F1=1.0000


## 5. Average các độ đo và độ đo theo từng class (câu e, f)

In [5]:
print("Nguyễn Đức Vinh - MSSV: 6451071087")
fold_results = pd.DataFrame(fold_rows)
class_results = pd.DataFrame(per_class_rows)
print("Kết quả từng fold:")
display(fold_results)
print("Average các độ đo trên 10 fold:")
display(fold_results[["Accuracy", "Precision", "Recall", "F1-score"]].mean().to_frame("Average"))
print("Average theo từng class:")
display(class_results.groupby("Class")[["Precision", "Recall", "F1-score"]].mean())

Nguyễn Đức Vinh - MSSV: 6451071087
Kết quả từng fold:


,Fold,Accuracy,Precision,Recall,F1-score
0,1,1.000000,1.000000,1.000000,1.000000
1,2,1.000000,1.000000,1.000000,1.000000
2,3,1.000000,1.000000,1.000000,1.000000
3,4,0.933333,0.952381,0.933333,0.937322
4,5,1.000000,1.000000,1.000000,1.000000
5,6,0.933333,0.916667,0.944444,0.922078
6,7,0.866667,0.916667,0.833333,0.841270
7,8,1.000000,1.000000,1.000000,1.000000
8,9,1.000000,1.000000,1.000000,1.000000
9,10,1.000000,1.000000,1.000000,1.000000


Average các độ đo trên 10 fold:


,Average
Accuracy,0.973333
Precision,0.978571
Recall,0.971111
F1-score,0.970067


Average theo từng class:


,Precision,Recall,F1-score
Class,,,
setosa,1.000000,1.000000,1.000000
versicolor,0.985714,0.933333,0.949883
virginica,0.950000,0.980000,0.960317


## 6. Chọn mô hình có F1-score tốt nhất và dự đoán 03 mẫu mới (câu g)

In [6]:
print("Nguyễn Đức Vinh - MSSV: 6451071087")
best_index = int(fold_results["F1-score"].to_numpy().argmax())
best_model = models[best_index]
print("Mô hình được chọn từ Fold:", best_index + 1)
new_samples = np.array([
    [6.2, 2.9, 4.3, 1.3],
    [5.1, 3.5, 1.4, 0.2],
    [7.3, 2.8, 6.4, 2.1]
], dtype="float32")
new_samples_scaled = new_samples / feature_max
new_predictions = np.argmax(best_model.predict(new_samples_scaled, verbose=0), axis=1)
for i, label in enumerate(new_predictions, start=1):
    print(f"Mẫu {i}: {class_names[int(label)]}")

Nguyễn Đức Vinh - MSSV: 6451071087
Mô hình được chọn từ Fold: 1
Mẫu 1: versicolor
Mẫu 2: setosa
Mẫu 3: virginica


> Ghi chú: do mỗi fold dùng tập validation khác nhau, một lớp có thể không xuất hiện trong một fold. Khi đó Precision/Recall/F1 của lớp đó được đặt bằng 0 theo `zero_division=0`, đúng với cách xử lý thống nhất của scikit-learn.